In [ ]:
# ============================================================
# CELL 1: Setup Environment and Configuration
# ============================================================
!pip install -q -U transformers accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Correct official Model ID for the 8B version
MODEL_ID = "mistralai/Ministral-8B-Instruct-2410"

# Securely fetch HF_TOKEN from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        print("⚠️ Warning: HF_TOKEN not found in Colab Secrets.")
except Exception:
    HF_TOKEN = None
    print("⚠️ Warning: Could not access Colab Secrets. Ensure the model is public or token is provided.")

print(f"✅ Ready to connect to: {MODEL_ID}")

In [ ]:
# ============================================================
# CELL 2: Lightweight Model Verification
# ============================================================

# We check if 'model' is already defined in the local variables
if 'model' in locals() and model is not None:
    print("✅ Model already loaded in memory. Skipping download to save time/VRAM.")
else:
    print("🚀 Model not found. Loading Ministral-8B now...")

    # Only define the config and load if necessary
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True,
        attn_implementation="sdpa",
        torch_dtype=torch.bfloat16
    )
    model.eval()
    print("✅ Ministral-8B loaded successfully!")

In [ ]:
# ============================================================
# CELL 3: Inference Test (Fixed)
# ============================================================
print("\nRunning test generation...")

test_prompt = "Explain why the sky is blue in one sentence."
messages = [{"role": "user", "content": test_prompt}]

# 1. Apply the template
# This returns a dictionary: {'input_ids': ..., 'attention_mask': ...}
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 2. Generate
with torch.no_grad():
    # 🔥 THE FIX IS HERE: Add '**' before inputs
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# 3. Decode
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nResponse: {decoded}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Running test generation...

Response: Explain why the sky is blue in one sentence.The sky appears blue because the atmosphere scatters blue light more than other colors due to the shorter wavelengths of blue light.


In [ ]:
# ============================================================
# CELL 3: Load your arXiv CSV and preview it
#
# TWO options for getting your CSV into Colab:
#
# OPTION A — Direct file upload (simplest):
#   from google.colab import files
#   uploaded = files.upload()          # opens a file picker
#   CSV_PATH = list(uploaded.keys())[0]
#
# OPTION B — Mount Google Drive (recommended for large files):
#   from google.colab import drive
#   drive.mount('/content/drive')
#   CSV_PATH = "/content/drive/MyDrive/your_folder/arxiv_papers.csv"
#
# Uncomment whichever option applies, then set CSV_PATH.
# ============================================================

import pandas as pd

# ── OPTION A: Direct upload ──────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = list(uploaded.keys())[0]

# ── OPTION B: Google Drive ───────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = "/content/drive/MyDrive/your_papers/arxiv_papers.csv"

# ── OPTION C: For quick testing — create a tiny synthetic CSV ─
import io
SAMPLE_CSV = """title,abstract,html_url
"Attention Is All You Need","We propose the Transformer, a model based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.","https://ar5iv.labs.arxiv.org/html/1706.03762"
"BERT: Pre-training of Deep Bidirectional Transformers","We introduce BERT, a new language representation model designed to pre-train deep bidirectional representations from unlabeled text.","https://ar5iv.labs.arxiv.org/html/1810.04805"
"""
df = pd.read_csv("results_fetch_cs_full.csv")
CSV_PATH = "sample_synthetic"   # just a label for option C

# ── Uncomment if using Option A or B and comment out Option C above ──
# df = pd.read_csv(CSV_PATH)

# ------------------------------------------------------------------
# Validate expected columns
# ------------------------------------------------------------------
REQUIRED_COLS = {"title", "abstract"}
missing = REQUIRED_COLS - set(df.columns)
if missing:
    raise ValueError(
        f"CSV is missing required columns: {missing}\n"
        f"Found columns: {list(df.columns)}"
    )

# Optional columns
HAS_HTML_URL   = "html_url"   in df.columns
HAS_FULL_TEXT  = "full_text"  in df.columns

print(f"✅ Loaded CSV: {CSV_PATH}")
print(f"   Rows       : {len(df)}")
print(f"   Columns    : {list(df.columns)}")
print(f"   html_url   : {'✅ present' if HAS_HTML_URL  else '❌ absent (will use abstract only)'}")
print(f"   full_text  : {'✅ present' if HAS_FULL_TEXT else '❌ absent'}")
print()

# Preview
df.head(3)


In [ ]:
# ============================================================
# CELL 4: HTML fetcher, boilerplate stripper, and the core
#         popular-science summarisation function
# ============================================================

import requests
from bs4 import BeautifulSoup
import re
import traceback

# ──────────────────────────────────────────────────────────────────
# 4-A  HTML fetcher (used when html_url column exists)
# ──────────────────────────────────────────────────────────────────

def fetch_html_body_content(html_url: str):
    """
    Fetch the body text of an arXiv HTML page (ar5iv / arxiv HTML format).
    Returns (text: str | None, status: str).
    """
    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"  ⚠️  HTML fetch failed: {e}")
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # Remove Abstract section (we use the structured abstract column instead)
    text = re.sub(
        r"Abstract[:\s].*?(?=(Introduction|1\.\s*Introduction))",
        "",
        text,
        flags=re.S | re.I,
    )

    # Truncate at Acknowledgements section
    for pat in [r"Acknowledgements?", r"ACKNOWLEDGEMENTS?"]:
        m = re.search(pat, text, re.I)
        if m:
            text = text[: m.start()]
            break

    text = "\n".join(line for line in text.splitlines() if line.strip())
    return text, "html_success"


# ──────────────────────────────────────────────────────────────────
# 4-B  arXiv boilerplate stripper
# ──────────────────────────────────────────────────────────────────

BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "×",
}

BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue", "Back to arXiv", "Why HTML?",
    "Report Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to Introduction", "Back to ", "Content selection saved",
    "Describe the issue below",
]


def get_article_snippet(full_text: str, max_chars: int = 4000) -> str:
    """
    Strip arXiv HTML navigation preamble; return the first `max_chars`
    characters of real article prose.
    """
    lines = full_text.splitlines()

    # Phase 1 — find where prose begins
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2 — filter residual boilerplate and bare section numbers
    cleaned = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            if cleaned:
                cleaned.append("")
            continue
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        if stripped in BOILERPLATE_EXACT:
            continue
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned.append(line)

    return "\n".join(cleaned)[:max_chars]

In [ ]:
#Load files in
import time

# 1. Load your specific input file
INPUT_FILE = "results_fetch_econ_full.csv"
df = pd.read_csv(INPUT_FILE)

# 2. Set the output column name (kept as 'llama_prompt' to match your snippet)
OUTPUT_COL = "mixtral_prompt"
df[OUTPUT_COL] = ""

# 3. Use T4/A100 device detection
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Loaded {len(df)} rows. Results will be saved to column: '{OUTPUT_COL}'")

In [ ]:
# --- Main evaluation loop for Ministral-8B ---
print(f"Starting batch processing of {len(df)} papers...")
print("=" * 50)

for index, row in df.iterrows():
    try:
        html_url = row.get("html_url", None)
        if not html_url or str(html_url) == "nan":
            continue

        # 1. Fetch the content (using your Cell 4 functions)
        article, status = fetch_html_body_content(html_url)
        if status != "html_success":
            print(f"Row {index}: HTML fetch failed ({status})")
            continue

        # 2. Get a larger snippet for a "Full" read
        # 10,000 chars is roughly 2,000-2,500 tokens
        article_snippet = get_article_snippet(article, max_chars=10000)

        # 3. Build chat message structure
        messages = [
            {"role": "user", "content": f"""You are a professional science popularization writer.

Task:
Rewrite the following scientific paper content into a clear and accessible explanation.

Requirements:
- Audience: general public with high-school background
- Length: 150–200 words
- Focus on the main idea, method, and contribution.
- Avoid equations and heavy jargon

Paper content:
{article_snippet}
"""}
        ]

        # 4. Tokenize with Ministral specifics
        encoded = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            truncation=True,
            max_length=6000 # Safety cap to prevent OOM
        ).to(device)

        # 5. Generate with unpacked dictionary
        # max_new_tokens=400 is plenty for a 200-word summary
        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_new_tokens=400,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id
            )

        # 6. Extract generated text
        input_length = encoded["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_length:]

        output_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # 7. Update DataFrame
        df.at[index, OUTPUT_COL] = output_text
        print(f"✅ Row {index}: Successfully processed")

        # Optional: Save progress every 5 rows in case of disconnect
        if (index + 1) % 5 == 0:
            df.to_csv("results_llama_phy_autosave.csv", index=False)

    except Exception as e:
        print(f"❌ Error processing Row {index}: {e}")
        # traceback.print_exc()
        continue

print("\n" + "=" * 50)
print("Batch processing complete.")
# Save the final results to CSV
OUTPUT_FILE = "results_mixtral_econ.csv"
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"🎉 Process Finished! Saved {len(df)} rows to {OUTPUT_FILE}")